# 🤖 Kraken Scalping Bot — BTC & SOL
**Strategy:** Smart Money Concepts (MSB + Order Blocks + FVG) + EMA + RSI + ATR stops  
**Pairs:** BTC/USD · BTC/USDT · SOL/USD · SOL/USDT

---
## 📋 Notebook phases

| Phase | Cells | Description |
|-------|-------|-------------|
| **1 — Setup** | 1–5 | Install, configure, connect to Kraken |
| **2 — Backtest** | 6–7 | Download history, run backtest, review results |
| **3 — Live/Paper loop** | 8–9 | Start the bot (only after reviewing backtest) |

> ⚠️ **Live trading is disabled by default.** See Cell 2 to enable it.  
> **Never share your API key. Keep WITHDRAW permission OFF on your Kraken key.**

---
## Phase 1 — Setup
*Run cells 1–5 once before anything else.*

In [1]:
# ── CELL 1 — Install dependencies (run once) ────────────────────────────────#
#import sys
#!{sys.executable} -m pip install ccxt pandas numpy matplotlib python-dotenv --quiet
#print('✅ Dependencies installed')

In [2]:
# ── CELL 2 — Configuration ───────────────────────────────────────────────────
#
# TRADING MODE
# ─────────────
# 'paper' → simulate trades, zero risk (DEFAULT — start here)
# 'live'  → real orders on Kraken
#
# To switch to live trading:
#   1. Change TRADING_MODE below to "live"
#   2. Enter your real API key and secret
#   3. Re-run this cell, then re-run the Kraken connection cell (Cell 5)
#   4. Make sure you have reviewed the backtest results first!
#
TRADING_MODE = "paper"   # <── change to "live" when ready

# ── Kraken API credentials ────────────────────────────────────────────────────
# Required permissions: Query Funds, Query Orders, Create Orders, Cancel Orders
# Keep 'Withdraw Funds' permission OFF for safety!
KRAKEN_API_KEY    = "your_api_key_here"
KRAKEN_API_SECRET = "your_api_secret_here"

# ── Pairs to trade ────────────────────────────────────────────────────────────
PAIRS = ["BTC/USD", "BTC/USDT", "SOL/USD", "SOL/USDT"]

# ── Strategy parameters ───────────────────────────────────────────────────────
STRATEGY_CONFIG = {
    "risk_percent":  1.0,    # % of equity risked per trade
    "rr":            3.0,    # risk-to-reward ratio for take-profit
    "use_trailing":  True,   # enable trailing stop
    "trail_percent": 0.25,   # trailing stop distance as % of price
    "commission":    0.026,  # Kraken taker fee %
    "slippage":      3,
}

# ── Bot settings ─────────────────────────────────────────────────────────────
CANDLE_LIMIT  = 200    # 1-min candles fetched per live cycle (min 50)
LOOP_INTERVAL = 60     # seconds between each live scan cycle
PAPER_EQUITY  = 10000  # starting virtual USD for paper mode

# ── Backtest settings ─────────────────────────────────────────────────────────
BACKTEST_CANDLES = 1000   # how many 1-min candles to backtest (max ~720 on Kraken free)
BACKTEST_PAIRS   = PAIRS  # which pairs to backtest (can be a subset)

# ── Mode banner ───────────────────────────────────────────────────────────────
if TRADING_MODE == "live":
    print("⚡ LIVE MODE — real orders will be placed on Kraken!")
    print("   Make sure you have reviewed the backtest results before running the loop.")
else:
    print("📄 PAPER MODE (default) — simulating trades, no real orders")
    print("   To switch to live: change TRADING_MODE to 'live' and re-run this cell.")
print(f"   Pairs: {', '.join(PAIRS)}")

📄 PAPER MODE (default) — simulating trades, no real orders
   To switch to live: change TRADING_MODE to 'live' and re-run this cell.
   Pairs: BTC/USD, BTC/USDT, SOL/USD, SOL/USDT


In [ ]:
# ── CELL 3 — Indicator functions ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timezone
import time, warnings
warnings.filterwarnings('ignore')

def ema(series, period):
    """Exponential Moving Average — fast/slow trend lines."""
    return series.ewm(span=period, adjust=False).mean()

def rsi(series, period=14):
    """
    RSI momentum filter.
    Long entries: RSI 52-80 (positive momentum, not overbought).
    Short entries: RSI 20-48 (negative momentum, not oversold).
    """
    delta    = series.diff()
    gain     = delta.clip(lower=0)
    loss     = (-delta).clip(lower=0)
    avg_gain = gain.ewm(com=period - 1, adjust=False).mean()
    avg_loss = loss.ewm(com=period - 1, adjust=False).mean()
    rs       = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def atr(high, low, close, period=10):
    """
    Average True Range — sets dynamic stop-loss distance.
    Shorter period = tighter stops, faster response to volatility.
    """
    prev_close = close.shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low  - prev_close).abs(),
    ], axis=1).max(axis=1)
    return tr.ewm(com=period - 1, adjust=False).mean()

def sma(series, period):
    return series.rolling(window=period).mean()

def highest(series, period):
    return series.rolling(window=period).max()

def lowest(series, period):
    return series.rolling(window=period).min()

print('✅ Indicator functions ready')

In [ ]:
# ── CELL 4 — Strategy signal engine ──────────────────────────────────────────
#
# Full SMC pipeline:
#   1. EMA-9/21 trend  2. RSI momentum  3. ATR volatility  4. Volume MA
#   5. Market Structure Breaks (MSB)  6. Order Blocks (OB)
#   7. Fair Value Gaps (FVG)  8. Entry signals  9. SL/TP levels

def compute_signals(df, config):
    """
    Run the full strategy on an OHLCV DataFrame.
    Returns (enriched_df, latest_signal_dict).
    Used by both the backtest engine and the live loop.
    """
    df = df.copy()

    # 1. Core indicators
    df['ema9']   = ema(df['close'], 9)
    df['ema21']  = ema(df['close'], 21)
    df['rsi']    = rsi(df['close'], 14)
    df['atr']    = atr(df['high'], df['low'], df['close'], 10)
    df['vol_ma'] = sma(df['volume'], 15)

    # 2. Market Structure Breaks
    # Higher high: current bar high > 15-bar rolling max of previous bars
    rolling_high = highest(df['high'].shift(1), 15).shift(1)
    rolling_low  = lowest(df['low'].shift(1), 15).shift(1)
    df['bull_break'] = (
        (df['high'] > rolling_high) &
        (df['close'] > df['open']) &
        (df['volume'] > df['vol_ma'] * 1.3)
    )
    df['bear_break'] = (
        (df['low'] < rolling_low) &
        (df['close'] < df['open']) &
        (df['volume'] > df['vol_ma'] * 1.3)
    )

    # 3. Order Blocks — candle just before the structure break
    bull_chg = df['bull_break'] & ~df['bull_break'].shift(1).fillna(False)
    bear_chg = df['bear_break'] & ~df['bear_break'].shift(1).fillna(False)
    df['bull_ob'] = pd.Series(
        np.where(bull_chg, df['low'].shift(1), np.nan), index=df.index).ffill()
    df['bear_ob'] = pd.Series(
        np.where(bear_chg, df['high'].shift(1), np.nan), index=df.index).ffill()

    # 4. Fair Value Gaps — price imbalance between candle[-2] and candle[0]
    df['fvg_bull'] = (df['low'].shift(2)  > df['high']) & (df['close'] > df['open'])
    df['fvg_bear'] = (df['high'].shift(2) < df['low'])  & (df['close'] < df['open'])

    # 5. Entry signals — all layers must align
    df['long_signal'] = (
        (df['bull_break'] | df['fvg_bull']) &
        (df['close'] > df['bull_ob']) &
        (df['close'] > df['ema9']) &
        (df['ema9']  > df['ema21']) &
        (df['rsi']   > 52) &
        (df['rsi']   < 80)
    )
    df['short_signal'] = (
        (df['bear_break'] | df['fvg_bear']) &
        (df['close'] < df['bear_ob']) &
        (df['close'] < df['ema9']) &
        (df['ema9']  < df['ema21']) &
        (df['rsi']   < 48) &
        (df['rsi']   > 20)
    )

    # 6. Exit levels — ATR-based SL, RR-scaled TP, trailing parameters
    df['stop_dist']    = df['atr'] * 0.6
    df['take_profit']  = df['stop_dist'] * config['rr']
    df['long_sl']      = df['close'] - df['stop_dist']
    df['long_tp']      = df['close'] + df['take_profit']
    df['short_sl']     = df['close'] + df['stop_dist']
    df['short_tp']     = df['close'] - df['take_profit']
    if config['use_trailing']:
        df['trail_points'] = df['close'] * (config['trail_percent'] / 100)
        df['trail_offset'] = df['stop_dist'] * 0.5
    else:
        df['trail_points'] = np.nan
        df['trail_offset'] = np.nan

    last = df.iloc[-1]
    signal = {
        'long_signal':  bool(last['long_signal']),
        'short_signal': bool(last['short_signal']),
        'close':        float(last['close']),
        'long_sl':      float(last['long_sl']),
        'long_tp':      float(last['long_tp']),
        'short_sl':     float(last['short_sl']),
        'short_tp':     float(last['short_tp']),
        'trail_points': float(last['trail_points']),
        'trail_offset': float(last['trail_offset']),
        'atr':          float(last['atr']),
        'rsi':          float(last['rsi']),
        'ema9':         float(last['ema9']),
        'ema21':        float(last['ema21']),
    }
    return df, signal

print('✅ Strategy engine ready')

In [ ]:
# ── CELL 5 — Kraken connection ───────────────────────────────────────────────
import ccxt

exchange = ccxt.kraken({
    'apiKey':          KRAKEN_API_KEY,
    'secret':          KRAKEN_API_SECRET,
    'enableRateLimit': True,
})

markets = exchange.load_markets()
print(f'✅ Connected to Kraken — {len(markets)} markets available')

# Validate pairs
valid_pairs = []
for pair in PAIRS:
    if pair in markets:
        valid_pairs.append(pair)
        print(f'  ✅ {pair}')
    else:
        print(f'  ⚠️  {pair} — not found on Kraken, skipped')
PAIRS = valid_pairs

# Show real balance only when using a real API key
if KRAKEN_API_KEY != 'your_api_key_here':
    try:
        bal  = exchange.fetch_balance()
        usd  = bal.get('USD',  {}).get('free', 0)
        usdt = bal.get('USDT', {}).get('free', 0)
        btc  = bal.get('XBT',  {}).get('free', 0)  # Kraken calls BTC 'XBT'
        sol  = bal.get('SOL',  {}).get('free', 0)
        print(f'\n💰 Live account balance:')
        print(f'   USD={usd}  USDT={usdt}  BTC={btc}  SOL={sol}')
    except ccxt.AuthenticationError:
        print('\n⚠️  Auth failed — check API key permissions')
else:
    print('\nℹ️  Placeholder API key — balance check skipped (paper mode)')

---
## Phase 2 — Backtest
*Run cells 6–7. Review results before starting the live loop.*  
*Cell 6 downloads historical data. Cell 7 runs the backtest and plots results.*  
*Both cells complete and return — there is no loop here.*

In [ ]:
# ── CELL 6 — Fetch historical OHLCV data for backtesting ────────────────────
#
# Downloads up to BACKTEST_CANDLES of 1-minute bars from Kraken.
# Kraken's free REST API allows ~720 candles per request on the 1m timeframe.
# For longer histories, increase the limit and use pagination (see note below).

def fetch_history(symbol, limit=720):
    """
    Fetch historical 1-min OHLCV bars from Kraken.
    Returns a clean DataFrame indexed by UTC timestamp.

    Note: Kraken allows ~720 bars per 1m request on the free tier.
    For longer backtests, use paginated fetching by passing `since`
    (a unix timestamp in ms) and looping until you have enough bars.
    """
    print(f'  Fetching {limit} candles for {symbol}...', end=' ')
    try:
        raw = exchange.fetch_ohlcv(symbol, timeframe='1m', limit=limit)
        df  = pd.DataFrame(raw, columns=['timestamp','open','high','low','close','volume'])
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
        df.set_index('timestamp', inplace=True)
        df = df.iloc[:-1]  # drop the still-forming last candle
        print(f'got {len(df)} bars')
        return df
    except Exception as e:
        print(f'ERROR: {e}')
        return pd.DataFrame()

# Download data for all backtest pairs
print('📥 Downloading historical data...')
historical_data = {}   # { symbol: DataFrame }
for pair in BACKTEST_PAIRS:
    df = fetch_history(pair, limit=BACKTEST_CANDLES)
    if not df.empty:
        historical_data[pair] = df

print(f'\n✅ Data ready for {len(historical_data)} pairs')
for sym, df in historical_data.items():
    span = f"{df.index[0].strftime('%Y-%m-%d %H:%M')} → {df.index[-1].strftime('%Y-%m-%d %H:%M')} UTC"
    print(f'   {sym}: {len(df)} bars  ({span})')

In [ ]:
# ── CELL 7 — Backtest engine + performance report ────────────────────────────
#
# Runs the full strategy bar-by-bar on the downloaded history.
# Simulates SL/TP exits (including trailing stop) and tracks equity.
# Prints a performance report and plots:
#   - Price chart with entry/exit markers
#   - Equity curve
#   - Drawdown curve
#   - Trade PnL distribution
#
# This cell runs ONCE and returns. No loop.

def run_backtest(df_raw, config, symbol=''):
    """
    Bar-by-bar backtest engine.

    Logic:
      - One position at a time (no pyramiding).
      - Entry at close of signal bar.
      - SL/TP checked against bar's high and low (TP takes priority if both hit).
      - Trailing stop: activates once price moves `trail_points` in our favour,
        then follows by `trail_offset` behind the extreme price.
      - Commission deducted on both entry and exit.

    Returns a dict with:
      trades    : list of trade dicts
      equity    : list of equity values over time (starts at 100)
      df        : enriched DataFrame with signal columns
    """
    if df_raw.empty or len(df_raw) < 50:
        print(f'  ⚠️  Not enough data for {symbol}')
        return None

    df, _ = compute_signals(df_raw, config)
    commission = config['commission'] / 100

    trades   = []
    equity   = [100.0]   # starts at 100 (percentage-based, easy to compare pairs)
    position = None

    for i in range(len(df)):
        row = df.iloc[i]
        eq  = equity[-1]

        # ── Manage open position ──────────────────────────────────────────────
        if position is not None:
            sl = position['sl']
            tp = position['tp']
            d  = position['direction']

            # Update trailing stop
            if config['use_trailing']:
                t_pts = position['trail_points']
                t_off = position['trail_offset']
                if d == 'long':
                    if (row['high'] - position['entry']) >= t_pts:
                        sl = max(sl, row['high'] - t_off)
                elif d == 'short':
                    if (position['entry'] - row['low']) >= t_pts:
                        sl = min(sl, row['low'] + t_off)
                position['sl'] = sl

            # Check exits (TP priority over SL on same bar)
            tp_hit   = (d == 'long'  and row['high'] >= tp) or \
                       (d == 'short' and row['low']  <= tp)
            stop_hit = (d == 'long'  and row['low']  <= sl) or \
                       (d == 'short' and row['high'] >= sl)

            if tp_hit or stop_hit:
                exit_price = tp if tp_hit else sl
                gross = (exit_price - position['entry']) / position['entry'] \
                        if d == 'long' \
                        else (position['entry'] - exit_price) / position['entry']
                net_pct = gross - commission   # exit commission
                new_eq  = eq * (1 + net_pct)
                equity.append(new_eq)
                trades.append({
                    'symbol':      symbol,
                    'direction':   d,
                    'entry_time':  position['entry_time'],
                    'exit_time':   df.index[i],
                    'entry_price': position['entry'],
                    'exit_price':  exit_price,
                    'result':      'TP' if tp_hit else 'SL',
                    'pnl_pct':     round(net_pct * 100, 4),
                    'equity':      round(new_eq, 4),
                })
                position = None
            else:
                equity.append(eq)   # no change this bar
        else:
            equity.append(eq)

        # ── Open new position (only when flat) ────────────────────────────────
        if position is None:
            if row['long_signal'] or row['short_signal']:
                d     = 'long' if row['long_signal'] else 'short'
                sl    = float(row['long_sl']  if d == 'long' else row['short_sl'])
                tp    = float(row['long_tp']  if d == 'long' else row['short_tp'])
                entry = float(row['close']) * (1 + commission if d == 'long' else 1 - commission)
                position = {
                    'direction':    d,
                    'entry':        entry,
                    'entry_time':   df.index[i],
                    'sl':           sl,
                    'tp':           tp,
                    'trail_points': float(row['trail_points']),
                    'trail_offset': float(row['trail_offset']),
                }

    return {'trades': trades, 'equity': equity[:len(df)], 'df': df}


def print_report(trades, symbol):
    """Print a formatted performance summary for one symbol."""
    if not trades:
        print(f'  {symbol}: no trades generated')
        return
    t    = pd.DataFrame(trades)
    wins = t[t['pnl_pct'] > 0]
    loss = t[t['pnl_pct'] <= 0]
    tp_c = (t['result'] == 'TP').sum()
    sl_c = (t['result'] == 'SL').sum()
    wr   = len(wins) / len(t) * 100
    pf   = abs(wins['pnl_pct'].sum() / loss['pnl_pct'].sum()) if len(loss) > 0 else float('inf')
    maxdd = 0
    peak  = t['equity'].iloc[0]
    for eq in t['equity']:
        if eq > peak:
            peak = eq
        dd = (peak - eq) / peak * 100
        if dd > maxdd:
            maxdd = dd
    print(f'  ┌─ {symbol} ' + '─' * (44 - len(symbol)))
    print(f'  │  Trades       : {len(t):>5}   (TP: {tp_c} / SL: {sl_c})')
    print(f'  │  Win rate     : {wr:>5.1f}%')
    print(f'  │  Total PnL    : {t["pnl_pct"].sum():>+6.2f}%')
    print(f'  │  Avg win      : {wins["pnl_pct"].mean():>+6.2f}%'   if len(wins) > 0 else '  │  Avg win      :    n/a')
    print(f'  │  Avg loss     : {loss["pnl_pct"].mean():>+6.2f}%'   if len(loss) > 0 else '  │  Avg loss     :    n/a')
    print(f'  │  Profit factor: {pf:>5.2f}')
    print(f'  │  Max drawdown : {maxdd:>5.2f}%')
    print(f'  └' + '─' * 50)


def plot_results(results_dict):
    """
    Plot a 4-panel performance dashboard for all backtested pairs.
    Panels:
      Top-left    : Price chart with long (▲) and short (▼) entry markers
      Top-right   : Equity curves (one line per pair, indexed to 100)
      Bottom-left : Drawdown curves
      Bottom-right: PnL distribution histogram
    """
    n_pairs = len(results_dict)
    if n_pairs == 0:
        print('No results to plot.')
        return

    colors  = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('Backtest Results — SMC Scalping Strategy', fontsize=14, fontweight='bold', y=1.01)
    ax_price, ax_equity, ax_dd, ax_dist = axes[0,0], axes[0,1], axes[1,0], axes[1,1]

    all_trades = []

    for idx, (symbol, res) in enumerate(results_dict.items()):
        if res is None:
            continue
        col    = colors[idx % len(colors)]
        df     = res['df']
        trades = res['trades']
        equity = res['equity']
        all_trades.extend(trades)

        # ── Panel 1: Price chart (first pair only to keep it readable) ────────
        if idx == 0:
            ax_price.plot(df.index, df['close'], color=col, linewidth=0.8, alpha=0.7, label='Close')
            ax_price.plot(df.index, df['ema9'],  color='orange', linewidth=0.8, linestyle='--', label='EMA 9', alpha=0.7)
            ax_price.plot(df.index, df['ema21'], color='blue',   linewidth=0.8, linestyle='--', label='EMA 21', alpha=0.7)
            # Plot entry signals
            longs  = df[df['long_signal']]
            shorts = df[df['short_signal']]
            ax_price.scatter(longs.index,  longs['close'],  marker='^', color='#4CAF50', s=60, zorder=5, label='Long entry')
            ax_price.scatter(shorts.index, shorts['close'], marker='v', color='#E91E63', s=60, zorder=5, label='Short entry')
            ax_price.set_title(f'Price + Signals ({symbol})')
            ax_price.legend(fontsize=7, loc='upper left')
            ax_price.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
            plt.setp(ax_price.xaxis.get_majorticklabels(), rotation=30, ha='right', fontsize=7)
            ax_price.set_ylabel('Price')
            ax_price.grid(True, alpha=0.2)

        # ── Panel 2: Equity curve ─────────────────────────────────────────────
        eq_series = pd.Series(equity, index=df.index[:len(equity)])
        ax_equity.plot(eq_series.index, eq_series.values, color=col, linewidth=1.2, label=symbol)

        # ── Panel 3: Drawdown ─────────────────────────────────────────────────
        rolling_max = eq_series.cummax()
        drawdown    = (eq_series - rolling_max) / rolling_max * 100
        ax_dd.fill_between(drawdown.index, drawdown.values, 0,
                           alpha=0.35, color=col, label=symbol)
        ax_dd.plot(drawdown.index, drawdown.values, color=col, linewidth=0.6)

    # ── Panel 4: PnL distribution ─────────────────────────────────────────────
    if all_trades:
        all_pnl = [t['pnl_pct'] for t in all_trades]
        wins_pnl = [p for p in all_pnl if p > 0]
        loss_pnl = [p for p in all_pnl if p <= 0]
        bins = 30
        ax_dist.hist(wins_pnl, bins=bins, color='#4CAF50', alpha=0.7, label=f'Wins ({len(wins_pnl)})')
        ax_dist.hist(loss_pnl, bins=bins, color='#E91E63', alpha=0.7, label=f'Losses ({len(loss_pnl)})')
        ax_dist.axvline(0, color='gray', linewidth=0.8, linestyle='--')
        ax_dist.set_title('PnL distribution (all pairs)')
        ax_dist.set_xlabel('PnL %')
        ax_dist.set_ylabel('Frequency')
        ax_dist.legend(fontsize=8)
        ax_dist.grid(True, alpha=0.2)

    # Finish equity and drawdown panels
    ax_equity.axhline(100, color='gray', linewidth=0.6, linestyle='--')
    ax_equity.set_title('Equity curve (base = 100)')
    ax_equity.set_ylabel('Equity')
    ax_equity.legend(fontsize=8)
    ax_equity.grid(True, alpha=0.2)
    ax_equity.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
    plt.setp(ax_equity.xaxis.get_majorticklabels(), rotation=30, ha='right', fontsize=7)

    ax_dd.axhline(0, color='gray', linewidth=0.6, linestyle='--')
    ax_dd.set_title('Drawdown %')
    ax_dd.set_ylabel('Drawdown %')
    ax_dd.legend(fontsize=8)
    ax_dd.grid(True, alpha=0.2)
    ax_dd.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
    plt.setp(ax_dd.xaxis.get_majorticklabels(), rotation=30, ha='right', fontsize=7)

    plt.tight_layout()
    plt.savefig('backtest_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('📊 Chart saved to backtest_results.png')


# ── Run the backtest ──────────────────────────────────────────────────────────
print('🔬 Running backtest...\n')
backtest_results = {}   # { symbol: result_dict }

for symbol, df_hist in historical_data.items():
    result = run_backtest(df_hist, STRATEGY_CONFIG, symbol=symbol)
    backtest_results[symbol] = result
    if result:
        print_report(result['trades'], symbol)

# ── Plot charts ───────────────────────────────────────────────────────────────
print('\n📈 Generating performance charts...')
plot_results(backtest_results)

# ── Final verdict ─────────────────────────────────────────────────────────────
all_trades_flat = [t for r in backtest_results.values() if r for t in r['trades']]
if all_trades_flat:
    total_pnl = sum(t['pnl_pct'] for t in all_trades_flat)
    total_wins = sum(1 for t in all_trades_flat if t['pnl_pct'] > 0)
    wr_all = total_wins / len(all_trades_flat) * 100
    print(f'\n─── COMBINED RESULTS ───────────────────────────────')
    print(f'  Total trades : {len(all_trades_flat)}')
    print(f'  Win rate     : {wr_all:.1f}%')
    print(f'  Total PnL    : {total_pnl:+.2f}%')
    print(f'────────────────────────────────────────────────────')

print('\n✅ Backtest complete. Review results above before starting the live loop.')
print('   When ready → run Phase 3 cells below.')

---
## Phase 3 — Live / Paper loop
⚠️ **Only run these cells after reviewing the backtest results above.**

- Cell 8 initialises the paper ledger and order helpers
- Cell 9 runs the loop — stop it with the **■ Interrupt Kernel** button
- Cell 10 shows the trade history table (run any time)

### Switching between paper and live
```python
# In Cell 2, change this line:
TRADING_MODE = "paper"   # safe default
TRADING_MODE = "live"    # real orders — use with caution
# Then re-run Cell 2 and Cell 5, then start the loop.
```

In [ ]:
# ── CELL 8 — Paper ledger + order helpers ────────────────────────────────────

class PaperLedger:
    """
    Virtual account for paper trading.
    Tracks equity, open positions, and completed trade history.
    Re-instantiated each time this cell runs to reset the simulation.
    """
    def __init__(self, starting_equity):
        self.equity    = starting_equity
        self.positions = {}   # { symbol: position_dict }
        self.trade_log = []

    def open_position(self, symbol, direction, price, qty, sl, tp, trail_pts, trail_off):
        cost = price * qty * (STRATEGY_CONFIG['commission'] / 100)
        self.equity -= cost
        self.positions[symbol] = {
            'direction': direction, 'entry': price, 'qty': qty,
            'sl': sl, 'tp': tp,
            'trail_points': trail_pts, 'trail_offset': trail_off,
            'trail_activated': False,
        }
        print(f"  📥 OPEN  {direction.upper():5s} {symbol} @ {price:.4f}"
              f" | SL={sl:.4f}  TP={tp:.4f}  Qty={qty:.6f}")

    def check_exit(self, symbol, high, low):
        if symbol not in self.positions:
            return
        pos = self.positions[symbol]
        sl, tp, d = pos['sl'], pos['tp'], pos['direction']
        if STRATEGY_CONFIG['use_trailing']:
            t, o = pos['trail_points'], pos['trail_offset']
            if d == 'long' and (high - pos['entry']) >= t:
                sl = max(sl, high - o)
            elif d == 'short' and (pos['entry'] - low) >= t:
                sl = min(sl, low + o)
            pos['sl'] = sl
        tp_hit   = (d == 'long' and high >= tp) or (d == 'short' and low  <= tp)
        stop_hit = (d == 'long' and low  <= sl) or (d == 'short' and high >= sl)
        if not (tp_hit or stop_hit):
            return
        exit_price = tp if tp_hit else sl
        qty = pos['qty']
        gross  = (exit_price - pos['entry']) * qty if d == 'long' \
                 else (pos['entry'] - exit_price) * qty
        net    = gross - exit_price * qty * (STRATEGY_CONFIG['commission'] / 100)
        self.equity += net
        icon = '✅' if tp_hit else '❌'
        tag  = 'TP' if tp_hit else 'SL'
        print(f"  📤 CLOSE {d.upper():5s} {symbol} @ {exit_price:.4f}"
              f" | {icon} {tag}  PnL={net:+.2f} USD  Equity=${self.equity:.2f}")
        self.trade_log.append({
            'symbol': symbol, 'direction': d,
            'entry': pos['entry'], 'exit': exit_price,
            'qty': qty, 'result': tag,
            'net_pnl_usd': round(net, 4), 'equity': round(self.equity, 2),
        })
        del self.positions[symbol]

    def summary(self):
        if not self.trade_log:
            print('No completed trades yet.')
            return None
        df   = pd.DataFrame(self.trade_log)
        wins = df[df['net_pnl_usd'] > 0]
        loss = df[df['net_pnl_usd'] <= 0]
        pf   = abs(wins['net_pnl_usd'].sum() / loss['net_pnl_usd'].sum()) \
               if len(loss) > 0 else float('inf')
        print('\n' + '═'*52)
        print('  📊 PAPER TRADING SUMMARY')
        print('═'*52)
        print(f'  Trades       : {len(df)}  (W:{len(wins)} / L:{len(loss)})')
        print(f'  Win rate     : {len(wins)/len(df)*100:.1f}%')
        print(f'  Total PnL    : ${df["net_pnl_usd"].sum():.2f}')
        print(f'  Profit factor: {pf:.2f}')
        print(f'  Equity now   : ${self.equity:.2f}')
        print('═'*52)
        return df


# Initialise ledger (resets virtual account each time this cell runs)
paper_ledger  = PaperLedger(PAPER_EQUITY)
live_positions = {}   # { symbol: { sl_order_id, tp_order_id, ... } }


# ── Data + sizing helpers ─────────────────────────────────────────────────────

def fetch_ohlcv(symbol, limit=CANDLE_LIMIT):
    """Fetch latest 1-min candles, drop the still-forming last bar."""
    try:
        raw = exchange.fetch_ohlcv(symbol, timeframe='1m', limit=limit + 1)
        df  = pd.DataFrame(raw, columns=['timestamp','open','high','low','close','volume'])
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
        df.set_index('timestamp', inplace=True)
        return df.iloc[:-1]
    except Exception as e:
        print(f'  ⚠️  Fetch error {symbol}: {e}')
        return pd.DataFrame()

def compute_qty(symbol, price, sl):
    """
    Risk-based position sizing:
      qty = (equity * risk_pct) / stop_distance
    Enforces Kraken minimum order sizes.
    """
    equity    = paper_ledger.equity if TRADING_MODE == 'paper' else _live_equity()
    stop_dist = abs(price - sl)
    if stop_dist == 0:
        return 0.0
    qty  = (equity * STRATEGY_CONFIG['risk_percent'] / 100) / stop_dist
    base = symbol.split('/')[0]
    mins = {'BTC': 0.0001, 'XBT': 0.0001, 'SOL': 0.5}
    return max(round(qty, 6), mins.get(base, 0.001))

def _live_equity():
    try:
        bal = exchange.fetch_balance()
        return float(bal.get('USD',  {}).get('free', 0) or 0) + \
               float(bal.get('USDT', {}).get('free', 0) or 0)
    except:
        return 0.0

def safe_cancel(order_id, symbol):
    try:
        exchange.cancel_order(order_id, symbol)
    except Exception:
        pass


# ── Paper execution ───────────────────────────────────────────────────────────

def execute_paper(symbol, signal, df):
    """Simulate paper entry/exit. Check exits first, then look for new entry."""
    if symbol in paper_ledger.positions and not df.empty:
        last = df.iloc[-1]
        paper_ledger.check_exit(symbol, float(last['high']), float(last['low']))
        return   # never open a new trade while one is open
    if signal['long_signal'] or signal['short_signal']:
        d   = 'long' if signal['long_signal'] else 'short'
        sl  = signal['long_sl']  if d == 'long' else signal['short_sl']
        tp  = signal['long_tp']  if d == 'long' else signal['short_tp']
        qty = compute_qty(symbol, signal['close'], sl)
        if qty > 0:
            paper_ledger.open_position(
                symbol, d, signal['close'], qty, sl, tp,
                signal['trail_points'], signal['trail_offset'])


# ── Live execution ────────────────────────────────────────────────────────────

def check_live_exit(symbol):
    """Poll Kraken to see if SL or TP order was filled; cancel the other."""
    pos = live_positions.get(symbol)
    if not pos:
        return
    try:
        sl_o = exchange.fetch_order(pos['sl_order_id'], symbol)
        tp_o = exchange.fetch_order(pos['tp_order_id'], symbol)
        if tp_o['status'] == 'closed':
            print(f'  ✅ LIVE TP HIT {symbol}')
            safe_cancel(pos['sl_order_id'], symbol)
            del live_positions[symbol]
        elif sl_o['status'] == 'closed':
            print(f'  ❌ LIVE SL HIT {symbol}')
            safe_cancel(pos['tp_order_id'], symbol)
            del live_positions[symbol]
    except Exception as e:
        print(f'  ⚠️  Exit check error {symbol}: {e}')

def execute_live(symbol, signal):
    """Place real market + SL limit + TP limit orders on Kraken."""
    if symbol in live_positions:
        check_live_exit(symbol)
        return
    if not (signal['long_signal'] or signal['short_signal']):
        return
    d     = 'long' if signal['long_signal'] else 'short'
    price = signal['close']
    sl    = signal['long_sl']  if d == 'long' else signal['short_sl']
    tp    = signal['long_tp']  if d == 'long' else signal['short_tp']
    qty   = compute_qty(symbol, price, sl)
    side  = 'buy'  if d == 'long' else 'sell'
    xside = 'sell' if d == 'long' else 'buy'
    if qty <= 0:
        return
    print(f'  ⚡ LIVE {d.upper()} {symbol} @ ~{price:.4f}  qty={qty}')
    try:
        entry_o = exchange.create_order(symbol, 'market', side, qty)
        sl_o    = exchange.create_order(symbol, 'stop_loss', xside, qty, sl,
                                        params={'ordertype': 'stop-loss', 'price': sl})
        tp_o    = exchange.create_order(symbol, 'limit', xside, qty, tp)
        live_positions[symbol] = {
            'direction': d, 'qty': qty, 'entry': price, 'sl': sl, 'tp': tp,
            'sl_order_id': sl_o['id'], 'tp_order_id': tp_o['id'],
        }
        print(f'    Entry={entry_o["id"]}  SL={sl_o["id"]}  TP={tp_o["id"]}')
    except ccxt.InsufficientFunds:
        print(f'  ⚠️  Insufficient funds for {symbol}')
    except Exception as e:
        print(f'  ⚠️  Order error {symbol}: {e}')


mode_label = '📄 PAPER' if TRADING_MODE == 'paper' else '⚡ LIVE'
print(f'✅ Execution helpers ready — mode: {mode_label}')
print(f'   Paper equity reset to ${PAPER_EQUITY:,.2f}')

In [ ]:
# ── CELL 9 — START THE BOT ───────────────────────────────────────────────────
# Runs continuously until you press ■ (Interrupt Kernel).
# Paper mode is the safe default — switch in Cell 2 to go live.

if TRADING_MODE == 'live':
    print('⚡ LIVE MODE — real orders will be placed on Kraken!')
    confirm = input("Type 'YES' to confirm: ")
    if confirm.strip() != 'YES':
        print('Aborted. Change TRADING_MODE back to "paper" or type YES to confirm.')
        raise SystemExit
else:
    print('📄 PAPER MODE — simulating trades, no real orders')

print(f'   Pairs: {PAIRS}')
print(f'   Loop every {LOOP_INTERVAL}s — stop with ■ Interrupt Kernel')
print('─' * 60)

cycle = 0
try:
    while True:
        cycle += 1
        now = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')
        print(f'\n{"═"*60}')
        print(f'  CYCLE #{cycle}  —  {now}')
        if TRADING_MODE == 'paper':
            print(f'  💰 Virtual equity: ${paper_ledger.equity:,.2f}')
        print('═'*60)

        for symbol in PAIRS:
            print(f'\n  📊 {symbol}')

            # 1. Fetch fresh candle data
            df = fetch_ohlcv(symbol)
            if df.empty:
                print('    ⚠️  No data — skipping')
                continue

            # 2. Compute strategy signals
            _, signal = compute_signals(df, STRATEGY_CONFIG)
            print(f"    Close={signal['close']:.4f}  RSI={signal['rsi']:.1f}  "
                  f"ATR={signal['atr']:.6f}  "
                  f"Long={'✅' if signal['long_signal'] else '❌'}  "
                  f"Short={'✅' if signal['short_signal'] else '❌'}")

            # 3. Execute (paper or live)
            if TRADING_MODE == 'paper':
                execute_paper(symbol, signal, df)
            else:
                execute_live(symbol, signal)

        # Print paper summary every 10 cycles
        if TRADING_MODE == 'paper' and cycle % 10 == 0:
            paper_ledger.summary()

        print(f'\n  ⏱  Sleeping {LOOP_INTERVAL}s...')
        time.sleep(LOOP_INTERVAL)

except KeyboardInterrupt:
    print('\n🛑 Bot stopped.')
    if TRADING_MODE == 'paper':
        paper_ledger.summary()
    elif live_positions:
        print(f'⚠️  Open live positions: {list(live_positions.keys())}')
        print('    Close them manually on Kraken!')

In [ ]:
# ── CELL 10 — View paper trade history (run any time) ────────────────────────
from IPython.display import display

trade_df = paper_ledger.summary()
if trade_df is not None:
    def color_pnl(val):
        if isinstance(val, (int, float)):
            return 'color: green; font-weight: bold' if val > 0 \
                   else 'color: red; font-weight: bold'
        return ''
    display(trade_df.style.applymap(color_pnl, subset=['net_pnl_usd']))